# Single-Pair EMD Benchmark — Julia EnergyFlow.jl

Benchmarks single EMD computation across backends, metrics, and event sizes.

In [1]:
# Setup
push!(LOAD_PATH, joinpath(@__DIR__, "..", "src"))
using EnergyFlow
using BenchmarkTools
using DelimitedFiles
using Printf

function load_event(path)
    readdlm(path, ',', Float64)
end

println("Julia $(VERSION), $(Threads.nthreads()) thread(s)")

Julia 1.12.5, 1 thread(s)


In [2]:
# Define test configs and run benchmarks
sizes = [(2,2),(10,10),(50,50),(100,100),(200,200),(500,500),(1000,1000),(2000,2000),(3000,3000),
         (100,200),(200,500),(500,1000),(500,2000),(1000,2000)]

backends = [:ns64, :ot64, :ns32, :ot32]

setups = [
    ("Euclidean_norm",   EuclideanMetric(), true),
    ("Euclidean_unnorm", EuclideanMetric(), false),
    ("EtaPhi_norm",      EtaPhiMetric(),    true),
]

results = []
for (n0, n1) in sizes
    ev0 = load_event(joinpath(@__DIR__, "..", "data", "event0_n$(n0).csv"))
    ev1 = load_event(joinpath(@__DIR__, "..", "data", "event1_n$(n1).csv"))
    for (setup_name, metric, norm) in setups
        for backend in backends
            emd(ev0, ev1; R=1.0, beta=1.0, norm=norm, backend=backend, metric=metric)  # warmup
            b = @benchmark emd($ev0, $ev1; R=1.0, beta=1.0, norm=$norm, backend=$backend, metric=$metric)
            push!(results, (n0=n0, n1=n1, setup=setup_name, backend=backend,
                           median_us=median(b.times)/1000, min_us=minimum(b.times)/1000,
                           allocs=b.allocs, memory=b.memory))
            @printf("%-5dx%-5d %-16s %-6s median=%10.1f \u00b5s  allocs=%d\n",
                    n0, n1, setup_name, backend, median(b.times)/1000, b.allocs)
        end
    end
end

2    x2     Euclidean_norm   ns64   median=       0.4 µs  allocs=57
2    x2     Euclidean_norm   ot64   median=       0.5 µs  allocs=62
2    x2     Euclidean_norm   ns32   median=       0.4 µs  allocs=57
2    x2     Euclidean_norm   ot32   median=       0.5 µs  allocs=62
2    x2     Euclidean_unnorm ns64   median=       0.5 µs  allocs=57
2    x2     Euclidean_unnorm ot64   median=       0.6 µs  allocs=62
2    x2     Euclidean_unnorm ns32   median=       0.5 µs  allocs=57
2    x2     Euclidean_unnorm ot32   median=       0.6 µs  allocs=62
2    x2     EtaPhi_norm      ns64   median=       0.4 µs  allocs=57
2    x2     EtaPhi_norm      ot64   median=       0.5 µs  allocs=62
2    x2     EtaPhi_norm      ns32   median=       0.4 µs  allocs=57
2    x2     EtaPhi_norm      ot32   median=       0.5 µs  allocs=62
10   x10    Euclidean_norm   ns64   median=       2.7 µs  allocs=57
10   x10    Euclidean_norm   ot64   median=       2.8 µs  allocs=62
10   x10    Euclidean_norm   ns32   median=     

In [3]:
# Save results
mkpath(joinpath(@__DIR__, "result"))
open(joinpath(@__DIR__, "result", "single_emd_julia.md"), "w") do io
    println(io, "# Single EMD Benchmark \u2014 Julia EnergyFlow.jl")
    println(io, "\nJulia $(VERSION), $(Threads.nthreads()) thread(s)\n")
    println(io, "| n0 | n1 | Setup | Backend | Median (\u00b5s) | Min (\u00b5s) | Allocs | Memory |")
    println(io, "|---|---|---|---|---|---|---|---|")
    for r in results
        @printf(io, "| %d | %d | %s | %s | %.1f | %.1f | %d | %d B |\n",
                r.n0, r.n1, r.setup, r.backend, r.median_us, r.min_us, r.allocs, r.memory)
    end
end
println("Results saved to result/single_emd_julia.md")

Results saved to result/single_emd_julia.md
